In [149]:
# Day 4

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn

import joblib
import os

import random

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [150]:
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive Mounted Successfully")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive Mounted Successfully


In [151]:
dataset_path = "/content/drive/MyDrive/food_dataset"
project_path = "/content/drive/MyDrive/SmartFood_Rescue"
model_path = project_path + "/models"
prediction_path = project_path + "/predictions"

print("Dataset Path:", dataset_path)
print("Project Path:", project_path)
print("Model Path:", model_path)
print("Prediction Path:", prediction_path)


Dataset Path: /content/drive/MyDrive/food_dataset
Project Path: /content/drive/MyDrive/SmartFood_Rescue
Model Path: /content/drive/MyDrive/SmartFood_Rescue/models
Prediction Path: /content/drive/MyDrive/SmartFood_Rescue/predictions


In [152]:

print("Available Model Files:\n")

for file in os.listdir(model_path):
    print(file)

print("\nAvailable Prediction Files:\n")

for file in os.listdir(prediction_path):
    print(file)


Available Model Files:

ann_regressor.pth
preprocessor.pkl
ann_classifier.pth
demand_thresholds.pkl

Available Prediction Files:

food_demand_predictions.csv
smartfood_predictions.csv


In [153]:
# Loading Preprocessor

preprocessor = joblib.load(
    model_path + "/preprocessor.pkl"
)

# Loading Demand Thresholds

thresholds = joblib.load(
    model_path + "/demand_thresholds.pkl"
)

low_threshold = thresholds["low_threshold"]
high_threshold = thresholds["high_threshold"]

print("Preprocessor Loaded Successfully!")
print("Demand Thresholds Loaded Successfully!")

print("\nLow Threshold:", low_threshold)
print("High Threshold:", high_threshold)

print("\nClass Mapping:")
print(thresholds["class_mapping"])


Preprocessor Loaded Successfully!
Demand Thresholds Loaded Successfully!

Low Threshold: 80.0
High Threshold: 243.0

Class Mapping:
{0: 'Low', 1: 'Medium', 2: 'High'}


In [154]:
# ANN Model Paths

regression_model_path = model_path + "/ann_regressor.pth"

classification_model_path = model_path + "/ann_classifier.pth"

print("Regression Model Exists:",
      os.path.exists(regression_model_path))

print("Classification Model Exists:",
      os.path.exists(classification_model_path))

Regression Model Exists: True
Classification Model Exists: True


In [155]:
train_data = pd.read_csv(dataset_path + "/train_prepared.csv")
X_sample = train_data.drop(columns=["id", "num_orders"])
X_sample_processed = preprocessor.transform(X_sample)


# Recreating ANN Regression Architecture

class ANNRegressor(nn.Module):
  def __init__(self):
    super(ANNRegressor, self).__init__()

    self.model = nn.Sequential(
        nn.Linear(X_sample_processed.shape[1], 128),
        nn.ReLU(),

        nn.Linear(128, 64),
        nn.ReLU(),

        nn.Linear(64, 1)
    )
  def forward(self, x):
    return self.model(x)

# Initialize The Model
ann_regressor = ANNRegressor()

# Load Saved Model Weights

ann_regressor.load_state_dict(torch.load(regression_model_path, map_location=device))

ann_regressor.to(device)

ann_regressor.eval()

print(ann_regressor)


ANNRegressor(
  (model): Sequential(
    (0): Linear(in_features=217, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [156]:
# Recreating ANN Classifier Architecture

class ANNClassifier(nn.Module):
  def __init__(self):
    super(ANNClassifier, self).__init__()

    self.model = nn.Sequential(
        nn.Linear(X_sample_processed.shape[1], 128),
        nn.ReLU(),

        nn.Linear(128, 64),
        nn.ReLU(),

        nn.Linear(64, 3)
    )

  def forward(self, x):
      return self.model(x)

# Initialize The Model

ann_classifier = ANNClassifier()

# Load Saved Model Weights

ann_classifier.load_state_dict(torch.load(classification_model_path, map_location=device))

ann_classifier.to(device)

ann_classifier.eval()

print(ann_classifier)

ANNClassifier(
  (model): Sequential(
    (0): Linear(in_features=217, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=3, bias=True)
  )
)


In [157]:
test_data = pd.read_csv(dataset_path + "/test_prepared.csv")
X_test = test_data.drop(columns=["id"])
X_test_processed = preprocessor.transform(X_test)

if hasattr(X_test_processed, "toarray"):
  X_test_processed = X_test_processed.toarray()

X_test_processed = X_test_processed.astype(np.float32)
X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32).to(device)

print("Test Data Loaded Successfully!")
print("Test Records:", X_test_tensor.shape[0])
print("Input Features:", X_test_tensor.shape[1])

Test Data Loaded Successfully!
Test Records: 32573
Input Features: 217


In [158]:
# Generating Predictions

ann_regressor.eval()
ann_classifier.eval()

with torch.no_grad():
  # ANN Regression
  regression_predictions = ann_regressor(X_test_tensor)

  # ANN Classification
  classification_outputs = ann_classifier(X_test_tensor)

  classification_predictions = torch.argmax(classification_outputs, dim=1)


In [159]:
# Regression Predictions

if isinstance(regression_predictions, torch.Tensor):
    regression_predictions = regression_predictions.detach().cpu().numpy()
else:
    regression_predictions = np.asarray(regression_predictions)

regression_predictions = regression_predictions.flatten()

# Reverse Log Transformation
regression_predictions = np.expm1(regression_predictions)

# Remove Negative Predictions
regression_predictions = np.maximum(regression_predictions, 0)


# Classification Predictions

if isinstance(classification_predictions, torch.Tensor):
    classification_predictions = classification_predictions.detach().cpu().numpy()
else:
    classification_predictions = np.asarray(classification_predictions)

classification_predictions = classification_predictions.flatten().astype(int)

In [160]:
predictions_df = pd.DataFrame({
    "id": test_data["id"],
    "Predicted_Orders": regression_predictions,
    "Predicted_Demands": classification_predictions
})

predictions_df["Predicted_Demands"] = predictions_df["Predicted_Demands"].map(thresholds["class_mapping"])

print(predictions_df.head(10))

        id  Predicted_Orders Predicted_Demands
0  1028232        267.104218            Medium
1  1127204        215.845520            Medium
2  1212707        147.012650            Medium
3  1082698         51.949425               Low
4  1400926         38.817257               Low
5  1284113        210.817245            Medium
6  1197966        228.106277            Medium
7  1132739        241.889282            Medium
8  1057981        114.191162            Medium
9  1095932        107.847992               Low


In [161]:
print("\nPrediction Validation")

print("Total Records:", len(predictions_df))

print("\nMissing Values:")
print(predictions_df.isnull().sum())

print("\nDemand Distribution:")
print(predictions_df["Predicted_Demands"].value_counts())

print("\nRegression Statistics:")
print(predictions_df["Predicted_Orders"].describe())


Prediction Validation
Total Records: 32573

Missing Values:
id                   0
Predicted_Orders     0
Predicted_Demands    0
dtype: int64

Demand Distribution:
Predicted_Demands
Medium    13524
Low        9953
High       9096
Name: count, dtype: int64

Regression Statistics:
count    32573.000000
mean       279.320312
std        457.377472
min         10.424193
25%         70.417763
50%        149.524063
75%        311.331543
max       9155.462891
Name: Predicted_Orders, dtype: float64


In [162]:
print("Infinite Values:")
print(np.isinf(predictions_df.select_dtypes(include=np.number)).sum())

print("\nFinal Prediction Sample:")
print(predictions_df.head(10))

Infinite Values:
id                  0
Predicted_Orders    0
dtype: int64

Final Prediction Sample:
        id  Predicted_Orders Predicted_Demands
0  1028232        267.104218            Medium
1  1127204        215.845520            Medium
2  1212707        147.012650            Medium
3  1082698         51.949425               Low
4  1400926         38.817257               Low
5  1284113        210.817245            Medium
6  1197966        228.106277            Medium
7  1132739        241.889282            Medium
8  1057981        114.191162            Medium
9  1095932        107.847992               Low


In [163]:
print("\nTop 10 Predicted Orders:")

print(
    predictions_df.nlargest(
        10, "Predicted_Orders"
    )
)


Top 10 Predicted Orders:
            id  Predicted_Orders Predicted_Demands
25712  1494011       9155.462891              High
22452  1093360       9101.916992              High
32124  1232035       8668.140625              High
28939  1295935       8623.963867              High
25713  1290319       8339.498047              High
22453  1338479       8304.375000              High
31990  1216601       8134.007812              High
28802  1431922       8069.284668              High
29711  1395446       7964.491211              High
26493  1116327       7939.659180              High


In [164]:
os.makedirs(prediction_path, exist_ok=True)

prediction_file = prediction_path + "/smartfood_predictions.csv"

predictions_df.to_csv(
    prediction_file,
    index=False
)

print("Predictions Saved Successfully!")

Predictions Saved Successfully!


In [165]:
saved_predictions = pd.read_csv(prediction_file)

print("Saved File Records:", len(saved_predictions))

print(saved_predictions.head())

Saved File Records: 32573
        id  Predicted_Orders Predicted_Demands
0  1028232        267.104220            Medium
1  1127204        215.845520            Medium
2  1212707        147.012650            Medium
3  1082698         51.949425               Low
4  1400926         38.817257               Low


In [166]:
assert len(predictions_df) == len(test_data)

assert predictions_df["id"].is_unique

assert predictions_df["Predicted_Orders"].notnull().all()

assert np.isfinite(predictions_df["Predicted_Orders"]).all()

assert (predictions_df["Predicted_Orders"] >= 0).all()

assert predictions_df["Predicted_Demands"].isin(
    ["Low", "Medium", "High"]
).all()

print("All Prediction Checks Passed!")

All Prediction Checks Passed!
